# MvDEC weighted-fusion ceiling on saved latents

Controlled diagnostic, label-guided: for every saved run, both views are centred and scaled to unit RMS,
then fused as `w*h1 + (1-w)*h2` and as a weighted concatenation for w = 0, 0.05, ..., 1.
The best w per run (chosen with labels) is an upper bound for fixed-weight linear fusion of these two views.

Input: a Kaggle dataset with `<group>/seed_*/artifact.pkl` from earlier MvDEC runs. Kaggle: GPU on, Internet on.

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)
assert gpus, "Enable a Kaggle GPU before running this diagnostic."

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

PINNED_COMMIT = "f1dd7e3e3123a400fb56416dde7e98e263a93987"
REPO = Path("/kaggle/temp/mvdec_fpmax")
RESULTS = Path("/kaggle/working/mvdec-fusion-weight-sweep")
for path in (REPO, RESULTS):
    if path.exists():
        shutil.rmtree(path)
RESULTS.mkdir(parents=True)
subprocess.run(["git", "clone", "-q", "-b", "claude/mvdec-fusion-weight-sweep",
                "https://github.com/vdat-18/mvdec_fpmax.git", str(REPO)], check=True)
subprocess.run(["git", "checkout", "-q", PINNED_COMMIT], cwd=REPO, check=True)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip() == PINNED_COMMIT
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_mvdec_fusion_weight_sweep.py"], cwd=REPO, check=True)

groups = sorted({p.parent.parent for p in Path("/kaggle/input").rglob("artifact.pkl")})
print("groups:", *groups, sep="\n  ")
assert groups, "Attach the dataset with <group>/seed_*/artifact.pkl"

In [ ]:
for group in groups:
    out = RESULTS / f"{group.name}.json"
    with (RESULTS / f"{group.name}.log").open("w", encoding="utf-8") as log:
        result = subprocess.run(
            [sys.executable, "scripts/mvdec_fusion_weight_sweep.py", str(group), "--output", str(out)],
            cwd=REPO, stdout=log, stderr=subprocess.STDOUT, check=False,
        )
    print(group.name, "exit code", result.returncode, flush=True)

In [ ]:
for path in sorted(RESULTS.glob("*.json")):
    data = json.loads(path.read_text())
    c = data["ceiling"]
    print(f"{path.stem}: paper fused {data['paper']['fused']}; "
          f"best NMI {c['best_nmi']['nmi']:.4f} (acc {c['best_nmi']['acc']:.4f}, {c['best_nmi']['fusion']}, "
          f"w={c['best_nmi']['w_view1']}, {c['best_nmi']['run']}); best ACC {c['best_acc']['acc']:.4f}; "
          f"points reaching paper {c['points_reaching_paper_fused']}/{c['points']}")
shutil.make_archive("/kaggle/working/mvdec-fusion-weight-sweep", "zip", RESULTS)